<a href="https://colab.research.google.com/github/Jaeji/FinalProject/blob/main/data_02_clusters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import os # Import os module to check for file existence

# 1. โหลดข้อมูล (เปลี่ยนชื่อไฟล์ตามที่คุณใช้)
file_name = "data_01_log_returns.csv"

# ตรวจสอบว่าไฟล์มีอยู่จริงหรือไม่
if not os.path.exists(file_name):
    print(f"ข้อผิดพลาด: ไม่พบไฟล์ '{file_name}' โปรดตรวจสอบว่าได้อัปโหลดไฟล์นี้ไปยังสภาพแวดล้อม Colab หรือไม่ และตรวจสอบชื่อไฟล์อีกครั้ง")
else:
    df = pd.read_csv(file_name, index_col=0)

    # 2. คำนวณ Correlation Matrix
    corr_matrix = df.corr()

    # 3. คำนวณ Distance Matrix โดยแปลงจาก Correlation
    distance_matrix = np.sqrt(np.maximum(0, 0.5 * (1 - corr_matrix)))
    condensed_distance = squareform(distance_matrix)

    # 4. ทำ Hierarchical Clustering (ใช้ Ward's method)
    Z = linkage(condensed_distance, method='ward')
    num_clusters = 4 # กำหนดจำนวนกลุ่มที่ต้องการ
    clusters = fcluster(Z, num_clusters, criterion='maxclust')

    # สร้าง DataFrame เก็บผลลัพธ์
    cluster_df = pd.DataFrame({
        'Cryptocurrency': corr_matrix.columns,
        'Cluster': clusters
    })

    # 5. ระบุเหรียญผู้นำ (Leader) ของแต่ละกลุ่ม
    leaders_list = []
    for c in range(1, num_clusters + 1):
        # ค้นหาเหรียญทั้งหมดที่อยู่ในกลุ่ม c
        coins_in_cluster = cluster_df[cluster_df['Cluster'] == c]['Cryptocurrency'].tolist()

        if len(coins_in_cluster) == 1:
            leader = coins_in_cluster[0]
        else:
            # ตัดตารางความสัมพันธ์มาเฉพาะเหรียญในกลุ่มเดียวกัน
            sub_corr = corr_matrix.loc[coins_in_cluster, coins_in_cluster]
            # คำนวณค่าเฉลี่ยความสัมพันธ์ (หักความสัมพันธ์ตัวเองที่เท่ากับ 1 ออก)
            mean_corrs = (sub_corr.sum(axis=1) - 1) / (len(coins_in_cluster) - 1)
            # เหรียญที่มีค่าเฉลี่ยความสัมพันธ์สูงสุดคือผู้นำ
            leader = mean_corrs.idxmax()

        # บันทึกสถานะว่าเหรียญไหนเป็นผู้นำ
        for coin in coins_in_cluster:
            is_leader = True if coin == leader else False
            leaders_list.append({'Cryptocurrency': coin, 'Is_Leader': is_leader})

    # นำสถานะผู้นำมารวมกับข้อมูลกลุ่ม
    leader_df = pd.DataFrame(leaders_list)
    final_df = pd.merge(cluster_df, leader_df, on='Cryptocurrency')

    # จัดเรียงข้อมูลตามกลุ่ม และให้ผู้นำขึ้นเป็นอันดับแรกในกลุ่มนั้นๆ
    final_df = final_df.sort_values(by=['Cluster', 'Is_Leader'], ascending=[True, False]).reset_index(drop=True)

    # 6. บันทึกผลลัพธ์ลงไฟล์ CSV
    final_df.to_csv("cluster_results_with_leaders.csv", index=False)
    print("จัดเก็บไฟล์ cluster_results_with_leaders.csv สำเร็จ")


จัดเก็บไฟล์ cluster_results_with_leaders.csv สำเร็จ
